# 🎬 YT Cloud Player

Download a YouTube video at **up to 1080p** on Google's cloud with [yt-dlp](https://github.com/yt-dlp/yt-dlp), then watch it right in this notebook.

**How to use:** click **Runtime → Run all** (or press ▶ on cells 1 and 2), then paste a link into the player.

- Everything runs on Colab's machine, nothing is installed on your computer.
- Files are saved to `/content/downloads` (see the 📁 Files panel). They disappear when the runtime shuts down, so use **Save to computer** or cell 4 to keep one.
- Only download videos you have the right to save; YouTube's terms restrict downloading.

In [ ]:
#@title ⚙️ 1 · Set up (≈30 s) { display-mode: "form" }
#@markdown Installs yt-dlp and Deno (yt-dlp needs a JavaScript runtime for YouTube). ffmpeg is already on Colab.
!pip install -q -U "yt-dlp[default]" deno
import shutil, yt_dlp
assert shutil.which("ffmpeg"), "ffmpeg missing"
print("✅ Ready — yt-dlp", yt_dlp.version.__version__, "· deno", "found" if shutil.which("deno") else "NOT found")

In [ ]:
#@title ▶️ 2 · Open the player { display-mode: "form" }
#@markdown Run this cell, then paste a YouTube link into the player that appears below it.
import os, threading, uuid, json, mimetypes, re
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from IPython.display import HTML, JSON, display
import yt_dlp

DL_DIR = "/content/downloads"
COOKIES = "/content/cookies.txt"
PORT = 8765
os.makedirs(DL_DIR, exist_ok=True)

# ---------- tiny file server with Range support (so the player can seek) ----------
class RangeHandler(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _resolve(self):
        name = os.path.basename(self.path.split("?")[0].lstrip("/"))
        from urllib.parse import unquote
        path = os.path.join(DL_DIR, unquote(name))
        return path if name and os.path.isfile(path) else None

    def _send(self, head_only):
        path = self._resolve()
        if not path:
            self.send_error(404)
            return
        size = os.path.getsize(path)
        start, end = 0, size - 1
        m = re.match(r"bytes=(\d*)-(\d*)", self.headers.get("Range", ""))
        if m and (m.group(1) or m.group(2)):
            if m.group(1):
                start = int(m.group(1))
                end = int(m.group(2)) if m.group(2) else size - 1
            else:  # suffix range: last N bytes
                start = max(0, size - int(m.group(2)))
            end = min(end, size - 1)
            self.send_response(206)
            self.send_header("Content-Range", f"bytes {start}-{end}/{size}")
        else:
            self.send_response(200)
        self.send_header("Content-Type", mimetypes.guess_type(path)[0] or "video/mp4")
        self.send_header("Accept-Ranges", "bytes")
        self.send_header("Content-Length", str(end - start + 1))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        if head_only:
            return
        with open(path, "rb") as f:
            f.seek(start)
            left = end - start + 1
            try:
                while left > 0:
                    chunk = f.read(min(1 << 20, left))
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    left -= len(chunk)
            except (BrokenPipeError, ConnectionResetError):
                pass  # the player closed this request while seeking

    def do_GET(self):
        self._send(False)

    def do_HEAD(self):
        self._send(True)


try:
    _srv = ThreadingHTTPServer(("0.0.0.0", PORT), RangeHandler)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
except OSError:
    pass  # already running from an earlier run of this cell

# ---------- downloader ----------
# Prefer H.264 + AAC at 1080p (plays everywhere), then any codec up to 1080p.
FORMAT = ("bv*[height<=1080][vcodec^=avc1]+ba[ext=m4a]/"
          "bv*[height<=1080]+ba/b[height<=1080]/b")
JOBS = {}


def _clean(msg):
    return re.sub(r"\x1b\[[0-9;]*m", "", str(msg)).replace("ERROR: ", "")


def _run(job_id, url):
    job = JOBS[job_id]

    def hook(d):
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
            done = d.get("downloaded_bytes", 0)
            # video and audio download one after the other; weight them 90/10
            part = 0 if job["parts_done"] == 0 else 90
            span = 90 if job["parts_done"] == 0 else 10
            if total:
                job["percent"] = round(part + span * done / total, 1)
            job["speed"] = (d.get("_speed_str") or "").strip()
            job["eta"] = (d.get("_eta_str") or "").strip()
            job["state"] = "downloading"
        elif d["status"] == "finished":
            job["parts_done"] += 1

    def pp_hook(d):
        if d["status"] == "started":
            job["state"] = "merging"

    opts = {
        "format": FORMAT,
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(DL_DIR, "%(title).80B [%(id)s].%(ext)s"),
        "restrictfilenames": True,
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
        "progress_hooks": [hook],
        "postprocessor_hooks": [pp_hook],
    }
    if os.path.exists(COOKIES):
        opts["cookiefile"] = COOKIES
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
            path = os.path.splitext(ydl.prepare_filename(info))[0] + ".mp4"
        job.update(
            state="done", percent=100, file=os.path.basename(path),
            title=info.get("title", ""), channel=info.get("uploader", ""),
            height=info.get("height"), fps=info.get("fps"),
            vcodec=(info.get("vcodec") or "").split(".")[0],
            duration=info.get("duration_string", ""),
            mb=round(os.path.getsize(path) / 1e6, 1),
        )
    except Exception as e:
        job.update(state="error", error=_clean(e))


def _start(url):
    url = (url or "").strip()
    if not re.match(r"https?://", url):
        return JSON({"error": "Paste a full link, starting with https://"})
    job_id = uuid.uuid4().hex[:8]
    JOBS[job_id] = {"state": "starting", "percent": 0, "parts_done": 0}
    threading.Thread(target=_run, args=(job_id, url), daemon=True).start()
    return JSON({"job": job_id})


def _status(job_id):
    return JSON(JOBS.get(job_id, {"state": "error", "error": "Unknown job"}))


def _library():
    files = sorted((f for f in os.listdir(DL_DIR) if f.endswith(".mp4")),
                   key=lambda f: os.path.getmtime(os.path.join(DL_DIR, f)), reverse=True)
    return JSON({"files": [{"file": f, "mb": round(os.path.getsize(os.path.join(DL_DIR, f)) / 1e6, 1)}
                           for f in files]})


def _save(name):
    from google.colab import files
    path = os.path.join(DL_DIR, os.path.basename(name))
    try:
        files.download(path)
        return JSON({"ok": True})
    except Exception as e:
        return JSON({"error": _clean(e)})


CHUNK = 2 * 1024 * 1024  # bytes per transfer when streaming through the notebook connection


def _size(name):
    path = os.path.join(DL_DIR, os.path.basename(name))
    return JSON({"size": os.path.getsize(path)} if os.path.isfile(path) else {"error": "File not found"})


def _chunk(name, offset):
    import base64
    path = os.path.join(DL_DIR, os.path.basename(name))
    with open(path, "rb") as f:
        f.seek(int(offset))
        return JSON({"b64": base64.b64encode(f.read(CHUNK)).decode("ascii")})


from google.colab import output
output.register_callback("ytc.size", _size)
output.register_callback("ytc.chunk", _chunk)
output.register_callback("ytc.start", _start)
output.register_callback("ytc.status", _status)
output.register_callback("ytc.library", _library)
output.register_callback("ytc.save", _save)

# ---------- the player UI ----------
display(HTML(r"""
<style>
#ytc{--bg:#0e1014;--card:#161a21;--line:#262c37;--text:#eef0f4;--muted:#8a93a6;--accent:#ff3b3b;--accent2:#ff7a45;--ok:#35d07f;
  font:15px/1.5 Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:var(--text);background:var(--bg);
  border-radius:18px;padding:26px;max-width:980px;box-shadow:0 10px 40px rgba(0,0,0,.35)}
#ytc *{box-sizing:border-box}
#ytc .hd{display:flex;align-items:center;gap:12px;margin-bottom:4px}
#ytc .logo{width:38px;height:38px;border-radius:11px;background:linear-gradient(135deg,var(--accent),var(--accent2));display:grid;place-items:center}
#ytc .logo svg{width:16px;height:16px;fill:#fff;margin-left:2px}
#ytc h1{font-size:21px;margin:0;font-weight:700;letter-spacing:-.01em}
#ytc .sub{color:var(--muted);font-size:13px;margin:0 0 18px 50px}
#ytc form{display:flex;gap:10px}
#ytc input{flex:1;min-width:0;background:var(--card);border:1px solid var(--line);color:var(--text);padding:13px 15px;border-radius:12px;font-size:15px;outline:none;transition:border-color .15s}
#ytc input:focus{border-color:var(--accent)}
#ytc button{border:0;border-radius:12px;padding:0 20px;font-weight:650;font-size:14px;cursor:pointer;color:#fff;background:linear-gradient(135deg,var(--accent),var(--accent2));transition:opacity .15s,transform .1s}
#ytc button:active{transform:scale(.97)}#ytc button:disabled{opacity:.45;cursor:default}
#ytc button.ghost{background:var(--card);border:1px solid var(--line);color:var(--text);padding:9px 14px;font-weight:550}
#ytc .prog{margin-top:16px;display:none}
#ytc .track{height:8px;background:var(--line);border-radius:4px;overflow:hidden}
#ytc .fill{height:100%;width:0;border-radius:4px;background:linear-gradient(90deg,var(--accent),var(--accent2));transition:width .35s ease}
#ytc .fill.pulse{animation:ytcp 1.1s ease-in-out infinite alternate}
@keyframes ytcp{from{opacity:.55}to{opacity:1}}
#ytc .msg{display:flex;justify-content:space-between;color:var(--muted);font-size:13px;margin-top:7px;gap:10px}
#ytc .err{margin-top:14px;padding:12px 14px;border-radius:12px;background:rgba(255,59,59,.1);border:1px solid rgba(255,59,59,.35);color:#ffb3b3;font-size:13px;white-space:pre-wrap;display:none}
#ytc .stage{margin-top:20px;display:none}
#ytc video{width:100%;aspect-ratio:16/9;background:#000;border-radius:14px;display:block}
#ytc .meta{display:flex;justify-content:space-between;align-items:flex-start;gap:14px;margin-top:12px;flex-wrap:wrap}
#ytc .title{font-weight:650;font-size:16px}
#ytc .chips{display:flex;gap:6px;flex-wrap:wrap;margin-top:6px}
#ytc .chip{font-size:12px;padding:2px 9px;border-radius:999px;background:var(--card);border:1px solid var(--line);color:var(--muted)}
#ytc .chip.hd{color:var(--ok);border-color:rgba(53,208,127,.4)}
#ytc .actions{display:flex;gap:8px}
#ytc h2{font-size:12px;color:var(--muted);text-transform:uppercase;letter-spacing:.08em;margin:26px 0 10px;font-weight:600}
#ytc ul{list-style:none;margin:0;padding:0;display:grid;gap:6px}
#ytc li{display:flex;justify-content:space-between;gap:10px;padding:10px 13px;border-radius:10px;background:var(--card);border:1px solid var(--line);cursor:pointer;font-size:13px;transition:border-color .15s}
#ytc li:hover{border-color:var(--accent)}#ytc li span:first-child{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#ytc li span:last-child{color:var(--muted);white-space:nowrap}
#ytc .empty{color:var(--muted);font-size:13px}
</style>
<div id="ytc">
  <div class="hd"><div class="logo"><svg viewBox="0 0 10 12"><path d="M0 0l10 6-10 6z"/></svg></div><h1>YT Cloud Player</h1></div>
  <p class="sub">Downloads up to 1080p on Google's cloud with yt-dlp, then plays it right here.</p>
  <form id="ytc-f"><input id="ytc-u" type="url" placeholder="Paste a YouTube link&hellip;" autocomplete="off" required><button id="ytc-go">Download</button></form>
  <div class="prog" id="ytc-p"><div class="track"><div class="fill" id="ytc-fill"></div></div>
    <div class="msg"><span id="ytc-m">Starting&hellip;</span><span id="ytc-s"></span></div></div>
  <div class="err" id="ytc-e"></div>
  <div class="stage" id="ytc-stage"><video id="ytc-v" controls playsinline preload="metadata"></video>
    <div class="meta"><div><div class="title" id="ytc-t"></div><div class="chips" id="ytc-c"></div></div>
      <div class="actions"><button class="ghost" id="ytc-save">&#11015; Save to computer</button></div></div></div>
  <h2>Downloaded this session</h2><ul id="ytc-lib"></ul>
</div>
<script>
(() => {
const PORT = """ + str(PORT) + r""";
const $ = id => document.getElementById(id);
const call = async (fn, args=[]) => (await google.colab.kernel.invokeFunction(fn, args, {})).data['application/json'];
let base = null, current = null, proxyBroken = false;
const videoBase = async () => base || (base = (await google.colab.kernel.proxyPort(PORT, {cache:false})).replace(/\/?$/, '/'));
const esc = s => String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
function showErr(t){ const e=$('ytc-e'); e.textContent=t||''; e.style.display=t?'block':'none'; }

const blobs = {};  // file -> blob: URL already loaded into the browser

// Fast path: stream through Colab's port proxy. Needs a third-party cookie,
// which Safari/Brave (and some Chrome settings) block -> resolves false.
function tryProxy(v, url, ms=8000) {
  return new Promise(res => {
    const done = ok => { clearTimeout(t); v.onloadedmetadata = v.onerror = null; res(ok); };
    const t = setTimeout(() => done(false), ms);
    v.onloadedmetadata = () => done(true);
    v.onerror = () => done(false);
    v.src = url;
  });
}

// Fallback: pull the file through the notebook connection in chunks and play
// it from a Blob in this page. Works in every browser; seeking works too.
async function loadViaKernel(file) {
  if (blobs[file]) return blobs[file];
  const { size, error } = await call('ytc.size', [file]);
  if (error) throw new Error(error);
  const CH = """ + str(CHUNK) + r""";
  const n = Math.ceil(size / CH), parts = new Array(n);
  let next = 0, got = 0;
  $('ytc-p').style.display = 'block'; $('ytc-fill').classList.remove('pulse');
  const worker = async () => {
    while (next < n) {
      const i = next++;
      const { b64 } = await call('ytc.chunk', [file, i * CH]);
      const bin = atob(b64), u8 = new Uint8Array(bin.length);
      for (let k = 0; k < bin.length; k++) u8[k] = bin.charCodeAt(k);
      parts[i] = u8; got += u8.length;
      const pct = Math.floor(got * 100 / size);
      $('ytc-fill').style.width = pct + '%';
      $('ytc-m').textContent = `Loading into player\u2026 ${pct}%`;
      $('ytc-s').textContent = `${(got / 1e6).toFixed(0)} / ${(size / 1e6).toFixed(0)} MB`;
    }
  };
  await Promise.all([worker(), worker(), worker()]);
  $('ytc-m').textContent = 'Ready'; $('ytc-s').textContent = '';
  return (blobs[file] = URL.createObjectURL(new Blob(parts, { type: 'video/mp4' })));
}

async function play(file, info={}) {
  current = file;
  const v = $('ytc-v');
  $('ytc-stage').style.display = 'block'; showErr('');
  $('ytc-t').textContent = info.title || file.replace(/\.mp4$/, '').replace(/_/g, ' ');
  const chips = [];
  if (info.height) chips.push(`<span class="chip ${info.height>=1080?'hd':''}">${info.height}p${info.fps?' \u00B7 '+info.fps+'fps':''}</span>`);
  if (info.vcodec) chips.push(`<span class="chip">${esc(info.vcodec)}</span>`);
  if (info.duration) chips.push(`<span class="chip">${esc(info.duration)}</span>`);
  if (info.mb) chips.push(`<span class="chip">${info.mb} MB</span>`);
  if (info.channel) chips.push(`<span class="chip">${esc(info.channel)}</span>`);
  $('ytc-c').innerHTML = chips.join('');
  google.colab.output.setIframeHeight(0, true);

  let ok = false;
  if (blobs[file]) { v.src = blobs[file]; ok = true; }
  else if (!proxyBroken) {
    try { ok = await tryProxy(v, (await videoBase()) + encodeURIComponent(file)); } catch (e) {}
    if (!ok) proxyBroken = true;  // don't wait on the proxy again this session
  }
  if (!ok) {
    v.removeAttribute('src'); v.load();
    try { v.src = await loadViaKernel(file); }
    catch (e) { showErr('Could not load the video: ' + e.message); return; }
  }
  v.onerror = () => showErr("Your browser can't decode this video's format (" + (info.vcodec || 'unknown codec') +
    "). Try Chrome, or use Save to computer and open it in a video player.");
  if (current === file) v.play().catch(() => {});
  google.colab.output.setIframeHeight(0, true);
}

async function loadLib() {
  const { files } = await call('ytc.library');
  $('ytc-lib').innerHTML = files.length ? '' : '<div class="empty">Nothing yet.</div>';
  files.forEach(f => {
    const li = document.createElement('li');
    li.innerHTML = `<span>${esc(f.file)}</span><span>${f.mb} MB</span>`;
    li.onclick = () => play(f.file, { mb: f.mb });
    $('ytc-lib').appendChild(li);
  });
  google.colab.output.setIframeHeight(0, true);
}

$('ytc-f').onsubmit = async e => {
  e.preventDefault(); showErr('');
  $('ytc-go').disabled = true; $('ytc-p').style.display = 'block';
  $('ytc-fill').style.width = '0'; $('ytc-fill').classList.add('pulse');
  $('ytc-m').textContent = 'Fetching video info\u2026'; $('ytc-s').textContent = '';
  const r = await call('ytc.start', [$('ytc-u').value]);
  if (r.error) { showErr(r.error); $('ytc-go').disabled = false; $('ytc-p').style.display = 'none'; return; }
  const timer = setInterval(async () => {
    const s = await call('ytc.status', [r.job]);
    if (s.state === 'downloading') {
      $('ytc-fill').classList.remove('pulse');
      $('ytc-fill').style.width = (s.percent || 0) + '%';
      $('ytc-m').textContent = `Downloading\u2026 ${Math.floor(s.percent || 0)}%`;
      $('ytc-s').textContent = [s.speed, s.eta && ('ETA ' + s.eta)].filter(Boolean).join(' \u00B7 ');
    } else if (s.state === 'merging') {
      $('ytc-fill').style.width = '100%'; $('ytc-fill').classList.add('pulse');
      $('ytc-m').textContent = 'Merging video + audio\u2026'; $('ytc-s').textContent = '';
    } else if (s.state === 'done') {
      clearInterval(timer); $('ytc-fill').classList.remove('pulse'); $('ytc-fill').style.width = '100%';
      $('ytc-m').textContent = `Done \u00B7 ${s.height || '?'}p`; $('ytc-go').disabled = false;
      play(s.file, s); loadLib();
    } else if (s.state === 'error') {
      clearInterval(timer); $('ytc-go').disabled = false; $('ytc-p').style.display = 'none';
      let msg = s.error || 'Download failed.';
      if (/not a bot|Sign in to confirm/i.test(msg)) msg += '\n\nYouTube is blocking Colab\'s IP. Run cell 3 to upload your cookies.txt, then try again.';
      showErr(msg);
    }
    google.colab.output.setIframeHeight(0, true);
  }, 800);
};

$('ytc-save').onclick = async () => {
  if (!current) return;
  $('ytc-save').disabled = true; $('ytc-save').textContent = 'Preparing\u2026';
  const r = await call('ytc.save', [current]);
  $('ytc-save').disabled = false; $('ytc-save').textContent = '\u2B07 Save to computer';
  if (r.error) showErr('Save failed: ' + r.error + '\nYou can also find it in the Files panel (left sidebar) under downloads/.');
};

loadLib();
})();
</script>
"""))


### Getting *"Sign in to confirm you're not a bot"*?
YouTube sometimes blocks Google's cloud IPs. Export your YouTube cookies as **cookies.txt** (Netscape format — for example with the *Get cookies.txt LOCALLY* browser extension while signed in to YouTube), run cell 3 to upload it, then retry in the player. No need to re-run cell 2.

In [ ]:
#@title 🍪 3 · Upload cookies.txt (only if YouTube blocks you) { display-mode: "form" }
from google.colab import files
import shutil
up = files.upload()
if up:
    name = next(iter(up))
    shutil.move(name, "/content/cookies.txt")
    print(f"✅ Saved {name} as /content/cookies.txt — try the download again.")

In [ ]:
#@title 💾 4 · Copy downloads to Google Drive (optional) { display-mode: "form" }
#@markdown Copies everything in `/content/downloads` to this Drive folder:
drive_folder = "YT Cloud Player"  #@param {type:"string"}
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
dest = os.path.join("/content/drive/MyDrive", drive_folder)
os.makedirs(dest, exist_ok=True)
for f in os.listdir("/content/downloads"):
    if f.endswith(".mp4") and not os.path.exists(os.path.join(dest, f)):
        shutil.copy2(os.path.join("/content/downloads", f), dest)
        print("Copied", f)
print("✅ Done:", dest)